# Hero run: SOLPS-ITER (B2.5 + EIRENE), DIII-D, flat uniform start to convergence

Reference run for the APS-DPP 2026 comparison: EIRENE at every step, dt = 1e-5 s, 15 000 steps from the step-50 state (the first 50 steps from the flat state were taken by the 3.0.8 executable; the training-dump build cannot start from the perfectly flat state). Run on Mora, 14-15 October 2026, 64 ranks, as 15 chained segments of 1000 steps so that a full plasma state exists every 1000 steps.

Data: `~/data/eirene_nn/reference_uniform_dt1e-5/` (states `ref_stepNNNNN`, histories `ref_b2time_NNNNN.nc`, segment logs, chain log). Mesh geometry: the SOLSTICE mesh store.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, netCDF4
from pathlib import Path
import hero_tools as ht
RUN = ht.DEFAULT_RUN
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
H = ht.histories(RUN)
print({k: (None if v is None else v.shape) for k, v in H.items()})

## Convergence: separatrix and target quantities against time

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
panels = [("nesepm", "n_e at the separatrix, midplane [m$^{-3}$]"), ("tesepm", "T_e at the separatrix, midplane [eV]"),
          ("tisepm", "T_i at the separatrix, midplane [eV]"), ("fnixap", "ion flux to the outer target [1/s]"),
          ("fnixip", "ion flux to the inner target [1/s]"), ("temxap", "peak T_e at the outer target [eV]")]
sept = Path(RUN) / "sept2026_3.0.8_b2time.nc"
S = netCDF4.Dataset(sept) if sept.exists() else None
for ax, (k, label) in zip(axes.ravel(), panels):
    if H.get(k) is None:
        ax.set_visible(False); continue
    ax.plot(H["time"] * 1e3, H[k][:, 0], color="#145DA0", lw=1.5, label="hero run (training-dump build)")
    if S is not None and k in S.variables:
        ts = S.variables["timesa"][:]; vs = S.variables[k][:].reshape(len(ts), -1)[:, 0]
        ax.plot(ts * 1e3, vs, color="#8A94A6", lw=1.0, ls="--", label="September run (3.0.8 build)")
    ax.set_title(label); ax.set_xlabel("time [ms]"); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=9)
plt.tight_layout(); plt.show()

## How converged: change between successive checkpoints (1000 steps = 10 ms apart)

In [ ]:
import glob, re
state_files = sorted(glob.glob(str(Path(RUN) / "states" / "ref_step*")))
steps = [int(re.search(r"ref_step(\d+)$", f).group(1)) for f in state_files]
fields = [ht.state_fields(f) for f in state_files]
names = list(fields[0])
fig, ax = plt.subplots(figsize=(8, 5))
for name in names:
    d = [ht.rel_l2(fields[i][name], fields[i - 1][name]) for i in range(1, len(fields))]
    ax.semilogy(steps[1:], d, marker="o", label=name)
ax.set_xlabel("step"); ax.set_ylabel("relative L2 change over the previous 1000 steps"); ax.grid(alpha=0.3, which="both"); ax.legend()
ax.set_title("State-space convergence (interior cells)")
plt.show()
print("last interval:", {n: f"{ht.rel_l2(fields[-1][n], fields[-2][n]):.2e}" for n in names})

## Wall time per 1000-step segment (64 ranks on Mora, EIRENE every step)

In [ ]:
wt = ht.segment_walltimes(RUN)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([s for s, _ in wt], [t / 60 for _, t in wt], width=700, color="#145DA0")
ax.set_xlabel("segment end step"); ax.set_ylabel("minutes per 1000 steps"); ax.grid(alpha=0.3, axis="y")
ax.set_title(f"total {sum(t for _, t in wt) / 3600:.1f} h, mean {np.mean([t for _, t in wt]) / 1000:.2f} s per step")
plt.show()

## Final state on the mesh

In [ ]:
polys = ht.mesh_polygons()
final = fields[-1]
fig, axes = plt.subplots(1, 4, figsize=(20, 7))
for ax, (name, log) in zip(axes, [("n_D+ [m^-3]", True), ("Te [eV]", True), ("Ti [eV]", True), ("potential [V]", False)]):
    pc = ht.plot_field(ax, polys, final[name], title=f"{name}, step {steps[-1]}", log=log, cmap="viridis" if log else "coolwarm")
    plt.colorbar(pc, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## Target profiles and the transient: inner and outer target rows at several times

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pick = [0, 2, 5, 9, len(fields) - 1]
for i in pick:
    axes[0].semilogy(fields[i]["n_D+ [m^-3]"][:, 0], label=f"step {steps[i]}")
    axes[1].plot(fields[i]["Te [eV]"][:, -1], label=f"step {steps[i]}")
axes[0].set_title("inner target column (ix = 1): n_D+ vs iy"); axes[0].set_xlabel("iy"); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].set_title("outer target column (ix = 96): T_e [eV] vs iy"); axes[1].set_xlabel("iy"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Where the last 1000 steps still change the state

Relative change per cell between the last two checkpoints: shows which region is still settling (expected: the divertor legs).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 7))
for ax, name in zip(axes, ["n_D+ [m^-3]", "Te [eV]"]):
    a, b = fields[-1][name], fields[-2][name]
    rel = np.abs(a - b) / np.maximum(np.abs(b), 1e-300)
    pc = ht.plot_field(ax, polys, rel, title=f"|change| / value, {name}, steps {steps[-2]} to {steps[-1]}", log=True, cmap="magma", vmin=1e-4, vmax=1e-1)
    plt.colorbar(pc, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()